# Week 11: Model Context Protocol (MCP)

606363 Computing Systems for Data Science and AI — every code example from the lesson page, in order.

Lesson page: https://zmomani.github.io/ComputingSystemsDS-AI/lessons/week11-mcp.html

**How to use:** open this notebook in the course codespace (folder `notebooks/`) and run the cells from the top (Shift+Enter, or Run All). Later examples often reuse variables from earlier ones. Examples that call OpenAI or other APIs need your key as a Codespaces secret or in the `.env` file. For Ollama examples first run `bash scripts/start-ollama.sh` in the terminal. Model-training examples are much faster on a GPU: open this notebook in Google Colab and choose Runtime → Change runtime type → T4 GPU (in Colab add API keys with the 🔑 Secrets panel).

In [ ]:
# Only needed in Google Colab (the course codespace already has these)
%pip install -q mcp

In [ ]:
# Imports that some of the lesson's code fragments assume
import nest_asyncio
nest_asyncio.apply()   # lets asyncio.run() work inside a notebook

## Sample MCP server implementation (Python)

This code is a file of an application, not a notebook step. The cell saves it as `mcp_server.py`; run it from a terminal on your computer (or with `!python mcp_server.py` here, if it is a script).

In [ ]:
%%writefile mcp_server.py
from mcp.server import MCPServer   # mcp 1.x: from mcp.server.fastmcp import FastMCP as MCPServer

# Create a simple MCP server
mcp = MCPServer("FileReader", instructions="Reads files from the local filesystem")

# Define and implement a tool (the JSON schema is derived from the type hints)
@mcp.tool()
def read_file(path: str) -> str:
    """Reads content from a file."""
    try:
        with open(path, "r") as f:
            return f.read()
    except Exception as e:
        return f"error: {e}"

# Start the server (stdio transport by default)
if __name__ == "__main__":
    mcp.run()

## Run in your browser

**What an MCP server does with JSON-RPC messages (pure Python)**

In [ ]:
import json

TOOLS = {"add": {"description": "Add two numbers",
                 "inputSchema": {"type": "object", "properties": {"a": {"type": "number"}, "b": {"type": "number"}}, "required": ["a", "b"]}}}

def handle(message):
    req = json.loads(message)
    if req["method"] == "initialize":
        result = {"protocolVersion": "2025-06-18", "capabilities": {"tools": {}}, "serverInfo": {"name": "demo", "version": "1.0"}}
    elif req["method"] == "tools/list":
        result = {"tools": [{"name": n, **spec} for n, spec in TOOLS.items()]}
    elif req["method"] == "tools/call":
        args = req["params"]["arguments"]
        result = {"content": [{"type": "text", "text": str(args["a"] + args["b"])}], "isError": False}
    else:
        return json.dumps({"jsonrpc": "2.0", "id": req["id"], "error": {"code": -32601, "message": "Method not found"}})
    return json.dumps({"jsonrpc": "2.0", "id": req["id"], "result": result})

for i, (method, params) in enumerate([("initialize", {}), ("tools/list", {}),
                                       ("tools/call", {"name": "add", "arguments": {"a": 2, "b": 40}}), ("ping/nope", {})], 1):
    msg = json.dumps({"jsonrpc": "2.0", "id": i, "method": method, "params": params})
    print("->", msg)
    print("<-", handle(msg), "\n")

## Test code for the course environment

**A real MCP server with the official SDK (pip install "mcp[cli]")**

This code is a file of an application, not a notebook step. The cell saves it as `server.py`; run it from a terminal on your computer (or with `!python server.py` here, if it is a script).

In [ ]:
%%writefile server.py
# 1. A real MCP server with the official SDK (pip install "mcp[cli]")
#    Save as server.py and run: python server.py   (speaks JSON-RPC over stdio)
from mcp.server import MCPServer   # mcp 1.x: from mcp.server.fastmcp import FastMCP as MCPServer

mcp = MCPServer("FileReader")

@mcp.tool()
def read_file(path: str) -> str:
    """Reads content from a file."""
    try:
        with open(path, "r") as f:
            return f.read()
    except Exception as e:
        return f"error: {e}"

if __name__ == "__main__":
    mcp.run()  # stdio transport by default

**An MCP client that launches the server above, lists tools, and calls one**

In [ ]:
# 2. An MCP client that launches the server above, lists tools, and calls one
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def main():
    params = StdioServerParameters(command="python", args=["server.py"])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()                 # handshake + capability negotiation
            tools = await session.list_tools()         # discovery
            print([t.name for t in tools.tools])
            result = await session.call_tool("read_file", {"path": "server.py"})
            print(result.content[0].text[:80])

asyncio.run(main())

**Hand-build the JSON-RPC messages MCP exchanges, to see the wire format**

In [ ]:
# 3. Hand-build the JSON-RPC messages MCP exchanges, to see the wire format
import json

initialize = {"jsonrpc": "2.0", "id": 1, "method": "initialize",
              "params": {"protocolVersion": "2025-06-18",
                         "capabilities": {},
                         "clientInfo": {"name": "demo-host", "version": "0.1"}}}
list_tools = {"jsonrpc": "2.0", "id": 2, "method": "tools/list"}
call_tool  = {"jsonrpc": "2.0", "id": 3, "method": "tools/call",
              "params": {"name": "read_file", "arguments": {"path": "notes.txt"}}}

for msg in (initialize, list_tools, call_tool):
    print(json.dumps(msg))

**The USB-C argument in numbers: custom adapters vs MCP connections**

In [ ]:
# 4. The USB-C argument in numbers: custom adapters vs MCP connections
models, sources = 4, 6
print("custom integrations :", models * sources)   # every model x every source
print("with MCP            :", models + sources)   # each side implements the protocol once